# 02 · LangGraph core: state, reducers, edges and a ReAct graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/agent-frameworks/02_langgraph_core.ipynb) &nbsp; 📖 Concept deck: [LangGraph core](https://nilesh-salpe.github.io/visual-ai/agent-frameworks/02-langgraph-core/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a custom state reducer, a graph with a conditional edge, and the ReAct agent as a two-node graph with `ToolNode` and `tools_condition`.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with a scripted fake chat model, so no API key or GPU is needed. Run the **Install** cell first (pinned to LangChain 1.4.6 and LangGraph 1.2.15).

In [ ]:
# @title Install pinned packages
%pip install -q "langchain==1.4.6" "langchain-core==1.6.10" "langgraph==1.2.15"

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool

class ScriptedChatModel(BaseChatModel):
    """A stand-in for a real chat model: replies with the scripted messages in order (cycling).
    Swap in e.g. init_chat_model("anthropic:claude-sonnet-5-5") to use a real model."""
    script: list
    i: int = 0
    calls: int = 0
    @property
    def _llm_type(self):
        return "scripted"
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        msg = self.script[self.i % len(self.script)]
        self.i += 1; self.calls += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])
    def bind_tools(self, tools, **kwargs):
        return self          # a real model would attach the tool schemas to every request

def call(name, args, id):
    """Shorthand for one tool call inside an AIMessage."""
    return {"name": name, "args": args, "id": id}

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    return f"18°C and sunny in {city}"

import operator, re
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

## Task 1: A reducer that keeps the last k items

Each state key has a **reducer** that merges a node's update into the current value. The default overwrites; `operator.add` appends lists. Write `keep_last(k)`, returning a reducer `(left, right) -> list` that concatenates and keeps only the last `k` items (a crude sliding window for history).

In [ ]:
def keep_last(k):
    # TODO: return a function (left, right) -> list
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_keep_last(f):
    r = f(3)
    assert r([1, 2], [3, 4]) == [2, 3, 4] and r([], [1]) == [1]
    class S(TypedDict):
        log: Annotated[list, f(3)]
    g = StateGraph(S)
    for name in "abcd":
        g.add_node(name, lambda s, name=name: {"log": [name]})
    g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", "c"); g.add_edge("c", "d"); g.add_edge("d", END)
    assert g.compile().invoke({"log": []})["log"] == ["b", "c", "d"], "the graph should keep only the last 3 entries"

check("keep_last", _test_keep_last, keep_last)

## Task 2: Routing with a conditional edge

Build and **compile** a graph over the state below. Node `classify` sets `route` to `"math"` if the question contains a digit, else `"chat"`. A **conditional edge** from `classify` goes to node `math` (answer: `"sum=<sum of the integers in the question>"`) or node `chat` (answer: `"Hello!"`). Both end at `END`.

In [ ]:
class RouteState(TypedDict):
    question: str
    route: str
    answer: str

def build_router_graph():
    g = StateGraph(RouteState)
    # TODO: nodes classify, math, chat; START -> classify; conditional edge; math/chat -> END
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_build_router_graph(f):
    app = f()
    r = app.invoke({"question": "add 2 and 40", "route": "", "answer": ""})
    assert r["route"] == "math" and r["answer"] == "sum=42", r
    r = app.invoke({"question": "hi there", "route": "", "answer": ""})
    assert r["route"] == "chat" and r["answer"] == "Hello!", r
    assert {"classify", "math", "chat"} <= set(app.get_graph().nodes)

check("build_router_graph", _test_build_router_graph, build_router_graph)

## Task 3: The ReAct agent as a graph

Build the agent loop as a graph over `MessagesState` (its `messages` key uses the `add_messages` reducer, which appends and de-duplicates by id):
* node `agent`: call `model.bind_tools(tools).invoke(state["messages"])` and return `{"messages": [reply]}`;
* node `tools`: `ToolNode(tools)`, which runs every tool call in the last message;
* `START → agent`; a conditional edge from `agent` with `tools_condition` (goes to `"tools"` if there are tool calls, else to `END`); `tools → agent`.

In [ ]:
def build_react_graph(model, tools):
    g = StateGraph(MessagesState)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_build_react_graph(f):
    m = ScriptedChatModel(script=[AIMessage(content="", tool_calls=[call("get_weather", {"city": "Rome"}, "r1")]), AIMessage(content="Rome: 18°C.")])
    app = f(m, [get_weather])
    out = app.invoke({"messages": [HumanMessage("Weather in Rome?")]})
    kinds = [type(x).__name__ for x in out["messages"]]
    assert kinds == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"], kinds
    assert out["messages"][2].content == "18°C and sunny in Rome"
    assert {"agent", "tools"} <= set(app.get_graph().nodes)

check("build_react_graph", _test_build_react_graph, build_react_graph)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: A reducer that keeps the last k items
def keep_last_solution(k):
    def reducer(left, right):
        return (list(left or []) + list(right or []))[-k:]
    return reducer

check("keep_last (reference)", _test_keep_last, keep_last_solution)

In [ ]:
# @title Solution for Task 2: Routing with a conditional edge
def build_router_graph_solution():
    g = StateGraph(RouteState)
    g.add_node("classify", lambda s: {"route": "math" if re.search(r"\d", s["question"]) else "chat"})
    g.add_node("math", lambda s: {"answer": f"sum={sum(map(int, re.findall(r'\d+', s['question'])))}"})
    g.add_node("chat", lambda s: {"answer": "Hello!"})
    g.add_edge(START, "classify")
    g.add_conditional_edges("classify", lambda s: s["route"], {"math": "math", "chat": "chat"})
    g.add_edge("math", END); g.add_edge("chat", END)
    return g.compile()

check("build_router_graph (reference)", _test_build_router_graph, build_router_graph_solution)

In [ ]:
# @title Solution for Task 3: The ReAct agent as a graph
def build_react_graph_solution(model, tools):
    bound = model.bind_tools(tools)
    g = StateGraph(MessagesState)
    g.add_node("agent", lambda s: {"messages": [bound.invoke(s["messages"])]})
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)
    g.add_edge("tools", "agent")
    return g.compile()

check("build_react_graph (reference)", _test_build_react_graph, build_react_graph_solution)

---
## Experiments
Stream the per-node updates of the ReAct graph, and print the graph as Mermaid text (paste it into any Mermaid viewer).

In [ ]:
m = ScriptedChatModel(script=[AIMessage(content="", tool_calls=[call("get_weather", {"city": "Rome"}, "r1")]), AIMessage(content="Rome: 18°C.")])
app = build_react_graph_solution(m, [get_weather])
for update in app.stream({"messages": [HumanMessage("Weather in Rome?")]}, stream_mode="updates"):
    for node, value in update.items():
        print(f"{node:6} ->", [type(x).__name__ for x in value["messages"]])
print(app.get_graph().draw_mermaid())

---
## Takeaways
* State is a typed dict; each key has a **reducer** (default: overwrite). Nodes return **partial updates**, never the whole state.
* Edges are fixed, **conditional edges** call a routing function; `END` stops the run.
* The ReAct agent is just `agent ⇄ tools` with `tools_condition` deciding when to stop.